# Population and Unemployment Data

How can two wide statistical tables be prepared for analysis?

Reshape World Bank population and Eurostat unemployment data, preserve observation flags, and calculate unemployment gaps by sex.

In [ ]:
import pandas as pd

medals = pd.DataFrame({
    "country": ["United States", "China", "Japan"],
    "gold": [40, 40, 20],
    "silver": [44, 27, 12],
    "bronze": [42, 24, 13],
})
medals

In [ ]:
medals_long = medals.melt(id_vars="country", var_name="medal", value_name="count")
medals_long

In [ ]:
medals_long.pivot_table(index="country", columns="medal", values="count")

In [ ]:
wb = pd.read_csv("data/world_bank_population.csv", skiprows=4)

wb.head()

In [ ]:
wb.describe(include="all")

In [ ]:
pop_long = wb.drop(columns = ["Indicator Name", "Indicator Code", "Unnamed: 70"])
pop_long = pop_long.melt(id_vars= ["Country Name", "Country Code"], var_name= "year", value_name= "population")
pop_long.year = pop_long.year.astype("Int64")
pop_long = pop_long.dropna(subset=["population"])

print(pop_long)

In [ ]:
pop_long[pop_long["year"] == 2025].sort_values("population", ascending= False).head(20)

In [ ]:
countries = ["Ukraine", "Poland", "Slovak Republic", "Hungary", "Romania", "Moldova", "Belarus"]
years = [1990, 2000, 2010, 2020, 2025]
(pop_long[pop_long["Country Name"].isin(countries) & pop_long["year"].isin(years)]).pivot_table(
    index= "year",
    columns= "Country Name",
    values= "population"
).reset_index()

In [ ]:
countries = ["Ukraine", "Poland", "Slovak Republic", "Hungary", "Romania", "Moldova", "Belarus"]
years = [1990, 2000, 2010, 2020, 2025]
neighbours = (pop_long[pop_long["Country Name"].isin(countries) & pop_long["year"].isin(years)]).pivot_table(
    index= "year",
    columns= "Country Name",
    values= "population"
).reset_index()

In [ ]:
une = pd.read_csv("data/une_rt_a.tsv", sep="\t")

une.head()

In [ ]:
une.head()

In [ ]:
une = une.rename(columns={une.columns[0]: "codes"})
une

In [ ]:
une_long = une.melt(id_vars= "codes", var_name= "year", value_name= "value")
une_long[["freq", "age", "unit", "sex", "geo"]] = une_long["codes"].str.split(",", expand=True)
une_long.year = une_long.year.astype("Int64")

In [ ]:
une_long.describe(include="all")

In [ ]:
une_long.age.value_counts()

In [ ]:
une_long["number"] = pd.to_numeric(une_long["value"].str.split(expand = True)[0], errors = "coerce")
une_long["flag"] = une_long["value"].str.split(expand = True)[1].value_counts(dropna = False).sum()

In [ ]:
une_long

In [ ]:
by_sex = une_long.loc[(une_long["unit"] == "PC_ACT") & 
    (une_long["age"] == "Y15-74")].pivot_table(
    index = ["year", "geo"], 
    columns = "sex", 
    values = "number", aggfunc = "sum").reset_index()
by_sex["gap"] = by_sex["F"] - by_sex["M"]
widest_gap_2025 = by_sex.loc[by_sex.loc[by_sex["year"] == 2003]["gap"].idxmax(), "geo"]

In [ ]:
by_sex

In [ ]:
une_long[
    (une_long["age"] == "Y15-24") & 
    (une_long["year"] == 2025) &
    (une_long["unit"] == "THS_PER") &
    (une_long["sex"] == "T") &
    (~une_long["geo"].isin(["EU27_2020", "EA20", "EA21"]))
        ]["number"].sum() * 1000

In [ ]:
ai_disclosure = "I used Claude to rephrase and explain the task to me without providing the actual code. I also used it to debug Task 2.3; this helped me understand the difference between the double `=` sign and the single `=` sign and identify my mistake. A key aspect of using Claude was the ability to interact with it in real-time, allowing me to ask questions whenever I didn't understand something or got stuck."